# Module 4 - A Bigram Model That Writes

## What you'll build

Time to build your first model that actually generates text. The
**bigram model** is the simplest "real" language model: it assumes the
next character depends only on the *single* character right before it.

The recipe is pure counting:

1. Reuse the `CharTokenizer` from the warm-up to turn text into ids.
2. Build a `vocab x vocab` table `N` where `N[i, j]` counts how often
   character `j` followed character `i`. Start every count at 1
   ("add-one smoothing") so nothing is ever impossible.
3. Normalise each row so it sums to 1 -- now `P[i]` is a probability
   distribution over "what comes after character i".
4. To **generate**, start at some character, sample the next one from
   its row of `P`, move there, and repeat. A seeded random generator
   makes the output reproducible.

It won't write Shakespeare -- with only one character of memory it
produces babble -- but it's a genuine probabilistic model, and it sets
up everything that follows.

In [ ]:
import numpy as np
from reference.tokenizer.char_tokenizer import CharTokenizer

class BigramLM:
    def __init__(self, text):
        self.tokenizer = CharTokenizer(text)
        V = self.tokenizer.vocab_size
        # TODO: build N (V x V) starting at 1 (add-one smoothing),
        #       count each consecutive (current, next) pair in the text,
        #       then row-normalise into self.P (each row sums to 1).
        raise NotImplementedError

    def generate(self, n, seed):
        # TODO: with np.random.default_rng(seed), start at id 0 and draw
        #       n characters from the current row of self.P, returning the
        #       decoded string.
        raise NotImplementedError


## Reveal the reference solution

Run the hidden cell to load the reference `BigramLM` (it redefines the
class with the complete implementation).

In [ ]:
# Reference solution (single source of truth: reference/bigram/bigram.py)

"""A bigram character-level language model.

This is the simplest "real" language model: the probability of the next
character depends only on the single character right before it. We count
how often each character follows each other character, add a little
smoothing so nothing is ever impossible, and turn those counts into
probabilities. Sampling from those probabilities lets us generate text.
"""

import numpy as np

from reference.tokenizer.char_tokenizer import CharTokenizer


class BigramLM:
    """Count-based bigram model over characters.

    Builds a (vocab x vocab) matrix where row i, column j holds the
    probability that character j follows character i.
    """

    def __init__(self, text):
        # Build the vocabulary and char <-> id maps from the text.
        self.tokenizer = CharTokenizer(text)
        V = self.tokenizer.vocab_size

        # Start every pair count at 1 (Laplace / add-one smoothing) so no
        # transition has zero probability, even if it never appears.
        N = np.ones((V, V), dtype=np.float64)

        # Count each consecutive (current, next) character pair.
        ids = self.tokenizer.encode(text)
        for current, nxt in zip(ids, ids[1:]):
            N[current, nxt] += 1

        # Row-normalize so each row is a probability distribution that
        # sums to 1 (probabilities of the next char given the current one).
        self.N = N
        self.P = N / N.sum(axis=1, keepdims=True)

    def generate(self, n, seed):
        """Sample `n` characters from a fixed seed and return the string.

        Starts from character id 0 and repeatedly draws the next id from
        the current row of P. The seeded rng makes output reproducible.
        """
        rng = np.random.default_rng(seed)
        V = self.tokenizer.vocab_size

        current = 0
        ids = []
        for _ in range(n):
            current = rng.choice(V, p=self.P[current])
            ids.append(int(current))

        return self.tokenizer.decode(ids)


## Check your work

These asserts confirm each row of `P` is a valid probability
distribution (sums to 1), that seeded generation is reproducible, and
that every generated character is in the vocabulary.

In [ ]:
import numpy as np
_text = 'hello world\nthe quick brown fox jumps over the lazy dog'
_m = BigramLM(_text)

assert _m.P.shape == (_m.tokenizer.vocab_size, _m.tokenizer.vocab_size), 'P wrong shape'
assert np.allclose(_m.P.sum(axis=1), 1.0), 'rows of P must sum to 1'

_a = _m.generate(20, seed=42)
_b = _m.generate(20, seed=42)
assert _a == _b, 'seeded generation should be reproducible'
assert len(_a) == 20, 'should generate 20 characters'
assert all(ch in _m.tokenizer.vocab for ch in _a), 'generated chars out of vocab'
print('PASS', f'score=1.00  (sample: {_a!r})')
